# CodeLens — Free Cloud Backend on Google Colab

Run CodeLens's Python backend and open-weight AI model (Gemma 3 4B) with **free T4 GPU acceleration**.

### Instructions:
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected (CPU also works).
2. Click **Runtime > Run all**.
3. Copy the generated `trycloudflare.com` URL and connect it to your frontend!

In [ ]:
# 1. Install Ollama and cloudflared tunnel
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [ ]:
# 2. Start Ollama and download Gemma 3 4B model
import subprocess, time

subprocess.Popen(["ollama", "serve"])
time.sleep(4)

!ollama pull gemma3:4b || ollama pull qwen2.5:3b

In [ ]:
# 3. Install CodeLens package
!pip install -q git+https://github.com/manju342r/codlenz.git

In [ ]:
# 4. Launch Cloudflare Tunnel and start CodeLens server
import os, re, subprocess

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

backend_url = None
print("Starting secure cloud tunnel...")
for line in iter(tunnel.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        backend_url = match.group(0)
        print("\n" + "=" * 65)
        print(f"🚀 LIVE CLOUD BACKEND URL: {backend_url}")
        print(f"🔗 Open in Vercel: https://codlenz.vercel.app/?backend={backend_url}")
        print(f"🔗 Open in Hugging Face: https://manju342r-codlenz.hf.space/?backend={backend_url}")
        print("=" * 65 + "\n")
        break

# Start CodeLens demo server with CORS enabled
os.environ["CORS_ORIGIN"] = "*"
!codelens serve --port 8000